In [5]:
import pandas as pd
import numpy as np


# ============================================
# CARREGANDO DATASET
# ============================================

df = pd.read_csv('../data/processed/dataset_v1.csv')


In [6]:
colunas_texto = [
    'Area',
    'NomeArea',
    'GrupoOrigem',
    'NomeGrupoOrigem',
    'Origem',
    'NomeOrigem',
    'Sexo',
    'NomeProc',
    'Material',
    'Meio'
]

for coluna in colunas_texto:
    df[coluna] = (
        df[coluna]
        .astype(str)
        .str.strip()
        .str.upper()
    )

In [7]:
# ============================================
# CRIANDO DIMENSÕES
# ============================================

# =========================
# DIM_EXAME
# =========================

dim_exame = df[
    [
        'CodigoMatrix',
        'CodigoFaturamento',
        'NomeProc',
        'Material',
        'Meio'
    ]
].drop_duplicates()

dim_exame = dim_exame.reset_index(drop=True)

dim_exame['IDExame'] = dim_exame.index + 1


# =========================
# DIM_AREA
# =========================

dim_area = df[
    [
        'Area',
        'NomeArea'
    ]
].drop_duplicates()

dim_area = dim_area.reset_index(drop=True)

dim_area['IDArea'] = dim_area.index + 1


# =========================
# DIM_ORIGEM
# =========================

dim_origem = df[
    [
        'GrupoOrigem',
        'NomeGrupoOrigem',
        'Origem',
        'NomeOrigem'
    ]
].drop_duplicates()

dim_origem = dim_origem.reset_index(drop=True)

dim_origem['IDOrigem'] = dim_origem.index + 1


# =========================
# DIM_PACIENTE
# =========================

dim_paciente = df[
    [
        'Sexo'
    ]
].drop_duplicates()

dim_paciente = dim_paciente.reset_index(drop=True)

dim_paciente['IDPaciente'] = dim_paciente.index + 1


# ============================================
# REALIZANDO MERGES
# ============================================

# =========================
# MERGE DIM_EXAME
# =========================

df = df.merge(
    dim_exame,
    on=[
        'CodigoMatrix',
        'CodigoFaturamento',
        'NomeProc',
        'Material',
        'Meio'
    ],
    how='left'
)


# =========================
# MERGE DIM_AREA
# =========================

df = df.merge(
    dim_area,
    on=[
        'Area',
        'NomeArea'
    ],
    how='left'
)


# =========================
# MERGE DIM_ORIGEM
# =========================

df = df.merge(
    dim_origem,
    on=[
        'GrupoOrigem',
        'NomeGrupoOrigem',
        'Origem',
        'NomeOrigem'
    ],
    how='left'
)


# =========================
# MERGE DIM_PACIENTE
# =========================

df = df.merge(
    dim_paciente,
    on=['Sexo'],
    how='left'
)


# ============================================
# CRIANDO FATO_EXAMES
# ============================================

fato_exames = df[
    [
        'NumeroPedido',

        'IDExame',
        'IDArea',
        'IDOrigem',
        'IDPaciente',

        'DataSistema',
        'DataHoraPedido',
        'DataCriacaoAmostra',
        'HoraCriacaoAmostra',
        'DataHoraImpressao',
        'DataHoraColeta',
        'DataHoraTriagem',
        'DataHoraResultado',
        'DataHoraLiberacaoTecnica',
        'DataHoraLiberacaoClinica',
        'DataHoraCancelamento',
        'DataHoraRetiradaProc',

        'FlagLiberacaoClinicaAutomatica',
        'ResultadoRetificado'
    ]
]


# ============================================
# VALIDAÇÕES
# ============================================

print('==============================')
print('SHAPES')
print('==============================')

print(f'fato_exames: {fato_exames.shape}')
print(f'dim_exame: {dim_exame.shape}')
print(f'dim_area: {dim_area.shape}')
print(f'dim_origem: {dim_origem.shape}')
print(f'dim_paciente: {dim_paciente.shape}')


print('\n==============================')
print('VALIDAÇÃO IDS NULOS')
print('==============================')

print(
    fato_exames[
        [
            'IDExame',
            'IDArea',
            'IDOrigem',
            'IDPaciente'
        ]
    ].isnull().sum()
)


print('\n==============================')
print('VALIDAÇÃO DUPLICIDADES')
print('==============================')

print(f'dim_exame: {dim_exame.duplicated().sum()}')
print(f'dim_area: {dim_area.duplicated().sum()}')
print(f'dim_origem: {dim_origem.duplicated().sum()}')
print(f'dim_paciente: {dim_paciente.duplicated().sum()}')


# ============================================
# EXPORTANDO TABELAS
# ============================================

dim_exame.to_csv(
    '../data/curated/dim_exame.csv',
    index=False
)

dim_area.to_csv(
    '../data/curated/dim_area.csv',
    index=False
)

dim_origem.to_csv(
    '../data/curated/dim_origem.csv',
    index=False
)

dim_paciente.to_csv(
    '../data/curated/dim_paciente.csv',
    index=False
)

fato_exames.to_csv(
    '../data/curated/fato_exames.csv',
    index=False
)

print('\n==============================')
print('EXPORTAÇÃO FINALIZADA')
print('==============================')

SHAPES
fato_exames: (22805, 19)
dim_exame: (210, 6)
dim_area: (11, 3)
dim_origem: (13, 5)
dim_paciente: (2, 2)

VALIDAÇÃO IDS NULOS
IDExame       0
IDArea        0
IDOrigem      0
IDPaciente    0
dtype: int64

VALIDAÇÃO DUPLICIDADES
dim_exame: 0
dim_area: 0
dim_origem: 0
dim_paciente: 0

EXPORTAÇÃO FINALIZADA


In [8]:
print(dim_area.duplicated().sum())
print(dim_origem.duplicated().sum())
print(dim_exame.duplicated().sum())
print(dim_paciente.duplicated().sum())

0
0
0
0


In [9]:
print(dim_area.shape)
print(dim_origem.shape)
print(dim_exame.shape)
print(dim_paciente.shape)

(11, 3)
(13, 5)
(210, 6)
(2, 2)


In [10]:
dim_area.sort_values('NomeArea')

,Area,NomeArea,IDArea
9,ANAT,ANATOMIA,10
3,BIOM,BIOLOGIA MOLECULAR,4
7,CT,CITOMETRIA DE FLUXO,8
5,COAG,COAGULAÇÃO,6
6,GEN,GENÉTICA,7
0,HEM,HEMATOLOGIA GERAL,1
8,IMU,IMUNOLOGIA,9
10,EXT,LABORATÓRIOS EXTERNOS,11
4,MIC,MICROBIOLOGIA,5
1,QUIM,QUÍMICA CLÍNICA,2


In [11]:
dim_origem.sort_values('NomeOrigem')

,GrupoOrigem,NomeGrupoOrigem,Origem,NomeOrigem,IDOrigem
9,HOCSAMB,H EVANGELICO AMBULATORIO,EAC,HOCS AMBULATORIO CERRADO,10
4,HOCS,HOSPITAL EVANGELICO SOROCABA,ECC,HOCS CENTRO CIRURGICO,5
12,HOCS,HOSPITAL EVANGELICO SOROCABA,EHEM,HOCS HEMODINAMICA,13
10,HOCSAMB,H EVANGELICO AMBULATORIO,ELEX,HOCS LABORATORIO EXTERNO,11
6,HOCSAMB,H EVANGELICO AMBULATORIO,EMO,HOCS MEDICINA OCUPACIONAL,7
5,HOCS,HOSPITAL EVANGELICO SOROCABA,EP2,HOCS POSTO II,6
2,HOCS,HOSPITAL EVANGELICO SOROCABA,EP3,HOCS POSTO III,3
11,HOCS,HOSPITAL EVANGELICO SOROCABA,EP4,HOCS POSTO IV,12
3,HOCS,HOSPITAL EVANGELICO SOROCABA,EP6,HOCS POSTO VI,4
0,HOCS,HOSPITAL EVANGELICO SOROCABA,EPA,HOCS PRONTO ATENDIMENTO,1


In [12]:
fato_exames[
    [
        'IDExame',
        'IDArea',
        'IDOrigem',
        'IDPaciente'
    ]
].isnull().sum()

IDExame       0
IDArea        0
IDOrigem      0
IDPaciente    0
dtype: int64

In [13]:
fato_exames[['IDArea']].nunique()

IDArea    11
dtype: int64

In [14]:
dim_area['IDArea'].nunique()

11

In [15]:
fato_exames.shape

(22805, 19)